# GraphTrustBench v5 — Gemma 4 readout
Runs `paper/scripts/gemma_readout.py` on 4x L4 with vLLM from the competition wheelhouse. Outputs go to `/kaggle/working/readout`.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ.update(TRANSFORMERS_NO_TF='1', TRANSFORMERS_NO_FLAX='1', TRANSFORMERS_OFFLINE='1', VLLM_NO_USAGE_STATS='1',
                  VLLM_WORKER_MULTIPROC_METHOD='spawn', VLLM_ENGINE_READY_TIMEOUT_S='1200', NCCL_DEBUG='WARN',
                  PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
# Wheelhouse install copied from the official getting-started notebook.
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not WHEELHOUSE_DIR.exists():
    WHEELHOUSE_DIR = Path(glob.glob('/kaggle/input/**/vllm-*.whl', recursive=True)[0]).parent
for pth_pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pth_pattern):
        try: os.unlink(pth)
        except OSError: pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        try: os.symlink(w, tmp_whl / name)
        except OSError: shutil.copyfile(w, tmp_whl / name)
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
print('Wheelhouse installation complete.')

In [ ]:
MODE = 'main'   # 'pilot' (development tasks, before pre-registration) or 'main'
driver = r"""
import glob, hashlib, json, sys
# vLLM spawns workers that re-import this file; only the parent may run the setup and the readout.
if __name__ == "__main__":
    from pathlib import Path
    src = Path(glob.glob('/kaggle/input/**/gemma_readout.py', recursive=True)[0]).parent
    art = Path('/kaggle/working/art')
    layout = {'paper/data/pilot_2026-11-06.json': 'pilot_2026-11-06.json',
              'paper/data/heldout_full_2026-11-07.json': 'heldout_full_2026-11-07.json',
              'paper/results/2026-11-07_localization_v3/per_task.jsonl': 'dev_per_task.jsonl',
              'paper/results/2026-11-08_extension_v3/per_task.jsonl.gz': 'ext_per_task.jsonl.gz'}
    layout.update({f'paper/scripts/{p.name}': p.name for p in src.glob('*.py')})
    # Kaggle decompresses .gz uploads; restore the committed gzip path and check the committed content hash.
    if not (src / 'ext_per_task.jsonl.gz').exists():
        import gzip
        raw = (src / 'ext_per_task.jsonl').read_bytes()
        assert hashlib.sha256(raw).hexdigest() == '4b6531359ecd8769100af6170090bb02a06255835da47e9f2eaadcfd438aa67f'
        gz = art / 'paper/results/2026-11-08_extension_v3/per_task.jsonl.gz'
        gz.parent.mkdir(parents=True, exist_ok=True)
        gz.write_bytes(gzip.compress(raw))
        del layout['paper/results/2026-11-08_extension_v3/per_task.jsonl.gz']
    for dest, name in layout.items():
        (art / dest).parent.mkdir(parents=True, exist_ok=True)
        if not (art / dest).exists(): (art / dest).symlink_to(src / name)
    comp = Path(glob.glob('/kaggle/input/**/gemma-4-developer-agent/tasks.jsonl', recursive=True)[0]).parent
    want = json.loads((art / 'paper/data/heldout_full_2026-11-07.json').read_text())['tasks_sha256']
    assert hashlib.sha256((comp / 'tasks.jsonl').read_bytes()).hexdigest() == want, 'tasks.jsonl differs from the frozen selection'
    model = Path(glob.glob('/kaggle/input/models/**/gemma-4-31b-it-qat-w4a16-ct/**/config.json', recursive=True)[0]).parent
    print('art', src, 'comp', comp, 'model', model, flush=True)
    sys.path.insert(0, str(art / 'paper/scripts'))
    import gemma_readout
    gemma_readout.run(sys.argv[1], art, comp, model, Path('/kaggle/working/readout'))
"""
Path('/kaggle/working/driver.py').write_text(driver)
proc = subprocess.run([sys.executable, '/kaggle/working/driver.py', MODE])
print('exit', proc.returncode)
assert proc.returncode == 0